# 03 - Models

Train Linear Regression, XGBoost va iTransformer cho bai toan du bao `traffic_volume` 24 gio tiep theo.

## 1. Load du lieu da xu ly

In [ ]:
from pathlib import Path
import json
import random
import time
import warnings

import joblib
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.multioutput import MultiOutputRegressor

from xgboost import XGBRegressor

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.4f}'.format)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
candidate_processed_dirs = [
    Path.cwd() / '..' / 'data' / 'processed',
    Path.cwd() / 'data' / 'processed',
    Path.cwd() / 'time_series _' / 'data' / 'processed',
]

processed_dir = next((path.resolve() for path in candidate_processed_dirs if (path / 'feature_metadata.json').exists()), None)
if processed_dir is None:
    raise FileNotFoundError('Khong tim thay thu muc data/processed. Hay chay notebook 02 truoc.')

project_dir = processed_dir.parents[1]
model_dir = project_dir / 'results' / 'models'
history_dir = project_dir / 'results' / 'training_history'
model_dir.mkdir(parents=True, exist_ok=True)
history_dir.mkdir(parents=True, exist_ok=True)

with open(processed_dir / 'feature_metadata.json', 'r', encoding='utf-8') as f:
    metadata = json.load(f)

feature_cols = metadata['feature_cols']
target_col = metadata['target_col']
seq_len = metadata['seq_len']
pred_len = metadata['pred_len']

feature_scaler = joblib.load(processed_dir / 'feature_scaler.pkl')
target_scaler = joblib.load(processed_dir / 'target_scaler.pkl')

train_npz = np.load(processed_dir / 'train_windows.npz')
val_npz = np.load(processed_dir / 'validation_windows.npz')
test_npz = np.load(processed_dir / 'test_windows.npz')

X_train_seq = train_npz['X'].astype(np.float32)
y_train = train_npz['y'].astype(np.float32)
X_val_seq = val_npz['X'].astype(np.float32)
y_val = val_npz['y'].astype(np.float32)
X_test_seq = test_npz['X'].astype(np.float32)
y_test = test_npz['y'].astype(np.float32)

print(f'Processed dir: {processed_dir}')
print(f'Model dir: {model_dir}')
print(f'History dir: {history_dir}')
print(f'X_train_seq: {X_train_seq.shape}, y_train: {y_train.shape}')
print(f'X_val_seq: {X_val_seq.shape}, y_val: {y_val.shape}')
print(f'X_test_seq: {X_test_seq.shape}, y_test: {y_test.shape}')

## 2. Ham danh gia va luu lich su

In [ ]:
def inverse_target(y_scaled):
    y_2d = y_scaled.reshape(-1, 1)
    y_inv = target_scaler.inverse_transform(y_2d).reshape(y_scaled.shape)
    return y_inv

def regression_metrics(y_true_scaled, y_pred_scaled):
    y_true = inverse_target(np.asarray(y_true_scaled))
    y_pred = inverse_target(np.asarray(y_pred_scaled))
    y_true_flat = y_true.reshape(-1)
    y_pred_flat = y_pred.reshape(-1)
    rmse = mean_squared_error(y_true_flat, y_pred_flat) ** 0.5
    return {
        'mae': float(mean_absolute_error(y_true_flat, y_pred_flat)),
        'rmse': float(rmse),
        'r2': float(r2_score(y_true_flat, y_pred_flat)),
    }

def save_json(data, path):
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

def save_history(history, name):
    history_path = history_dir / f'{name}_history.json'
    save_json(history, history_path)
    return history_path

def summarize_model(name, train_metrics, val_metrics, test_metrics, train_seconds):
    return {
        'model': name,
        'train_seconds': float(train_seconds),
        'train': train_metrics,
        'validation': val_metrics,
        'test': test_metrics,
    }

## 3. Chuan bi du lieu cho model tabular

In [ ]:
# Linear Regression va XGBoost dung feature tai thoi diem cuoi cua input window
# de du bao vector 24 gio tiep theo.
X_train_tab = X_train_seq[:, -1, :]
X_val_tab = X_val_seq[:, -1, :]
X_test_tab = X_test_seq[:, -1, :]

print(X_train_tab.shape, X_val_tab.shape, X_test_tab.shape)

## 4. Train Linear Regression

In [ ]:
start_time = time.time()
linear_model = LinearRegression()
linear_model.fit(X_train_tab, y_train)
linear_train_seconds = time.time() - start_time

linear_pred_train = linear_model.predict(X_train_tab)
linear_pred_val = linear_model.predict(X_val_tab)
linear_pred_test = linear_model.predict(X_test_tab)

linear_train_metrics = regression_metrics(y_train, linear_pred_train)
linear_val_metrics = regression_metrics(y_val, linear_pred_val)
linear_test_metrics = regression_metrics(y_test, linear_pred_test)

linear_history = summarize_model(
    'linear_regression',
    linear_train_metrics,
    linear_val_metrics,
    linear_test_metrics,
    linear_train_seconds,
)

joblib.dump(linear_model, model_dir / 'linear_regression.pkl')
save_history(linear_history, 'linear_regression')
display(pd.DataFrame(linear_history).T)

## 5. Train XGBoost

In [ ]:
xgb_base = XGBRegressor(
    n_estimators=150,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    objective='reg:squarederror',
    random_state=SEED,
    n_jobs=-1,
    tree_method='hist',
)

xgb_model = MultiOutputRegressor(xgb_base, n_jobs=1)

start_time = time.time()
xgb_model.fit(X_train_tab, y_train)
xgb_train_seconds = time.time() - start_time

xgb_pred_train = xgb_model.predict(X_train_tab)
xgb_pred_val = xgb_model.predict(X_val_tab)
xgb_pred_test = xgb_model.predict(X_test_tab)

xgb_train_metrics = regression_metrics(y_train, xgb_pred_train)
xgb_val_metrics = regression_metrics(y_val, xgb_pred_val)
xgb_test_metrics = regression_metrics(y_test, xgb_pred_test)

xgb_history = summarize_model(
    'xgboost',
    xgb_train_metrics,
    xgb_val_metrics,
    xgb_test_metrics,
    xgb_train_seconds,
)

joblib.dump(xgb_model, model_dir / 'xgboost_multioutput.pkl')
save_history(xgb_history, 'xgboost')
display(pd.DataFrame(xgb_history).T)

## 6. Train iTransformer

In [ ]:
class ITransformer(nn.Module):
    def __init__(self, seq_len, pred_len, n_features, d_model=128, n_heads=8, num_layers=3, dropout=0.1):
        super().__init__()
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.n_features = n_features
        self.value_embedding = nn.Linear(seq_len, d_model)
        self.variable_embedding = nn.Parameter(torch.randn(1, n_features, d_model) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, pred_len),
        )

    def forward(self, x):
        # x: [batch, seq_len, n_features]
        x = x.permute(0, 2, 1)  # [batch, n_features, seq_len]
        x = self.value_embedding(x) + self.variable_embedding
        x = self.encoder(x)
        x = x.mean(dim=1)
        return self.head(x)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

batch_size = 256
epochs = 10
patience = 3
learning_rate = 1e-3

train_dataset = TensorDataset(torch.from_numpy(X_train_seq), torch.from_numpy(y_train))
val_dataset = TensorDataset(torch.from_numpy(X_val_seq), torch.from_numpy(y_val))
test_dataset = TensorDataset(torch.from_numpy(X_test_seq), torch.from_numpy(y_test))

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

itransformer_model = ITransformer(
    seq_len=seq_len,
    pred_len=pred_len,
    n_features=len(feature_cols),
    d_model=64,
    n_heads=4,
    num_layers=2,
    dropout=0.1,
).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(itransformer_model.parameters(), lr=learning_rate, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None, device='cpu'):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss = 0.0
    total_samples = 0
    preds = []
    targets = []
    
    with torch.set_grad_enabled(is_train):
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            
            if is_train:
                optimizer.zero_grad(set_to_none=True)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            if is_train:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            
            batch_size_now = X_batch.size(0)
            total_loss += loss.item() * batch_size_now
            total_samples += batch_size_now
            preds.append(y_pred.detach().cpu().numpy())
            targets.append(y_batch.detach().cpu().numpy())
    
    avg_loss = total_loss / total_samples
    preds = np.concatenate(preds, axis=0)
    targets = np.concatenate(targets, axis=0)
    return avg_loss, preds, targets

In [ ]:
best_val_loss = float('inf')
best_state = None
epochs_without_improvement = 0
itransformer_history = {
    'model': 'itransformer',
    'config': {
        'seq_len': seq_len,
        'pred_len': pred_len,
        'n_features': len(feature_cols),
        'batch_size': batch_size,
        'epochs': epochs,
        'patience': patience,
        'learning_rate': learning_rate,
        'device': str(device),
    },
    'epochs': [],
}

start_time = time.time()
for epoch in range(1, epochs + 1):
    train_loss, _, _ = run_epoch(itransformer_model, train_loader, criterion, optimizer, device)
    val_loss, val_pred_epoch, val_target_epoch = run_epoch(itransformer_model, val_loader, criterion, None, device)
    scheduler.step(val_loss)
    
    val_metrics_epoch = regression_metrics(val_target_epoch, val_pred_epoch)
    epoch_record = {
        'epoch': epoch,
        'train_loss': float(train_loss),
        'val_loss': float(val_loss),
        'val_mae': val_metrics_epoch['mae'],
        'val_rmse': val_metrics_epoch['rmse'],
        'val_r2': val_metrics_epoch['r2'],
        'lr': float(optimizer.param_groups[0]['lr']),
    }
    itransformer_history['epochs'].append(epoch_record)
    print(epoch_record)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {key: value.detach().cpu().clone() for key, value in itransformer_model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= patience:
            print(f'Early stopping tai epoch {epoch}')
            break

itransformer_train_seconds = time.time() - start_time

if best_state is not None:
    itransformer_model.load_state_dict(best_state)

train_loss_final, itransformer_pred_train, train_target_final = run_epoch(itransformer_model, train_loader, criterion, None, device)
val_loss_final, itransformer_pred_val, val_target_final = run_epoch(itransformer_model, val_loader, criterion, None, device)
test_loss_final, itransformer_pred_test, test_target_final = run_epoch(itransformer_model, test_loader, criterion, None, device)

itransformer_history['train_seconds'] = float(itransformer_train_seconds)
itransformer_history['best_val_loss'] = float(best_val_loss)
itransformer_history['final'] = {
    'train_loss': float(train_loss_final),
    'val_loss': float(val_loss_final),
    'test_loss': float(test_loss_final),
    'train': regression_metrics(train_target_final, itransformer_pred_train),
    'validation': regression_metrics(val_target_final, itransformer_pred_val),
    'test': regression_metrics(test_target_final, itransformer_pred_test),
}

torch.save({
    'model_state_dict': itransformer_model.state_dict(),
    'metadata': metadata,
    'model_config': itransformer_history['config'],
}, model_dir / 'itransformer.pt')
save_history(itransformer_history, 'itransformer')

display(pd.DataFrame(itransformer_history['epochs']))
display(pd.DataFrame(itransformer_history['final']).T)

## 7. Tong hop ket qua va luu lich su train

In [ ]:
summary = {
    'linear_regression': linear_history,
    'xgboost': xgb_history,
    'itransformer': itransformer_history,
}

summary_rows = []
summary_rows.append({
    'model': 'linear_regression',
    'train_seconds': linear_history['train_seconds'],
    **{f"validation_{k}": v for k, v in linear_history['validation'].items()},
    **{f"test_{k}": v for k, v in linear_history['test'].items()},
})
summary_rows.append({
    'model': 'xgboost',
    'train_seconds': xgb_history['train_seconds'],
    **{f"validation_{k}": v for k, v in xgb_history['validation'].items()},
    **{f"test_{k}": v for k, v in xgb_history['test'].items()},
})
summary_rows.append({
    'model': 'itransformer',
    'train_seconds': itransformer_history['train_seconds'],
    **{f"validation_{k}": v for k, v in itransformer_history['final']['validation'].items()},
    **{f"test_{k}": v for k, v in itransformer_history['final']['test'].items()},
})

summary_df = pd.DataFrame(summary_rows).sort_values('validation_rmse')
summary_df.to_csv(history_dir / 'model_summary.csv', index=False)
save_json(summary, history_dir / 'all_training_history.json')

display(summary_df)
print('Da luu models:')
for path in sorted(model_dir.iterdir()):
    print(path.name)
print('Da luu training history:')
for path in sorted(history_dir.iterdir()):
    print(path.name)